# Using a Self-Hosted Model via vLLM

InterviewPlayground can use a self-hosted model (served with
[vLLM](https://github.com/vllm-project/vllm)) as the participant
simulator, the InterviewReportCard judge, or both.

**Requirements:** a running vLLM server. This notebook does not start
one for you — run the `vllm serve` command below in a separate
terminal first, then run the cells here against it.

## 1. Start a local vLLM server

In a separate terminal (not in this notebook):

```bash
vllm serve <model-name> --host 0.0.0.0 --port 8000
```

In [ ]:
from interviewplayground import load_preset, set_default_model

## 2. Use the self-hosted model as the participant simulator

`api_key` must be a non-empty string — vLLM ignores its value but
requires one to be present.

In [ ]:
set_default_model(
    "openai/<model-name>",
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
)

study = load_preset("obesity_weight_management")
p = study.participants[0]
response = p.ask("How have weight conversations with your doctor gone?")
print(response)

Model-specific serving flags (e.g. a model's `chat_template_kwargs`)
go in `extra_body`, forwarded as-is to every completion call:

In [ ]:
set_default_model(
    "openai/Qwen/Qwen3.5-4B",
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

## 3. Use the self-hosted model as the InterviewReportCard judge

Pass the same `api_base`/`api_key` to `study.evaluate()` (or any
individual `evaluate_*` function):

In [ ]:
results = study.evaluate(
    model="openai/<judge-model-name>",
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
)
results.keys()

## 4. Batch evaluation against a local server

`use_batch=True` works against vLLM too — since a local server has no
provider batch API, it automatically falls back to running prompts
concurrently over `call()` instead. Concurrency defaults to the
`LLM_BATCH_CONCURRENCY` environment variable (16 if unset) — set it to
roughly your server's `--max-num-seqs`.

In [ ]:
import os
os.environ["LLM_BATCH_CONCURRENCY"] = "16"

results = study.evaluate(
    model="openai/<judge-model-name>",
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
    use_batch=True,
)